# NB-11-ENTSOG · Jupyter companion

Calculations use the private Workbench API, pinned to version **1.43.0**. You need your individual API key and the private input bundle supplied for **NB-11-ENTSOG**.

1. Open this notebook in a fresh Python 3.12 or 3.13 kernel. Keep `notebook-methods-v1.43.0.zip` beside it when working locally; otherwise setup downloads the checksum-verified public helper archive.
2. Run the helper setup cell, then select your private input directory or ZIP. Local Jupyter can use `EWB_NOTEBOOK_INPUT_DIR` or `EWB_NOTEBOOK_INPUT_ZIP`. Colab uploads send inputs to your Google runtime; use local Jupyter when the inputs must stay on your computer.
3. Run the teaching cells in order. Inspect inputs, receipts, tables and plots locally. Results are saved beneath `data/processed/notebook-results`; save the notebook privately after execution.

The download contains no saved execution outputs. [Previously reviewed figures and results](https://sherandoresearch.github.io/power-markets-series/NB-11-ENTSOG.html) remain available in the static reader. A successful calculation validates its numerical contract, not a scientific claim. Saved results retain their original version; these notebooks request version 1.43.0 explicitly.

The static reader preserves previously reviewed results; rerunning this notebook creates new private execution evidence.

<!-- book-volume-context -->
## Where this fits in the volumes

This is an observed gas-data application of the volume’s identity and revision principles. It distinguishes source-value revisions from additional hours acquired by the collector; it does not reproduce an electricity network model.

- [Volume V, Chapter 6 — Valid time, knowledge time, and revisions](https://sherandoresearch.github.io/power-markets-series/vol5_representing_power_markets.html#c6)
- [Volume V, §8.2 — The mapping contract](https://sherandoresearch.github.io/power-markets-series/vol5_representing_power_markets.html#c8-2)

In [ ]:
# Install the reviewed transport and notebook helpers. This cell makes no API calls.
import hashlib, importlib.metadata, io, json, os, pathlib, ssl, stat, subprocess, sys, tempfile, urllib.request, zipfile

if sys.version_info[:2] not in {(3, 12), (3, 13)}:
    raise RuntimeError("Use a Python 3.12 or 3.13 kernel for this reviewed release.")
helper_name = "notebook-methods-v1.43.0.zip"
expected_sha256 = "2bea8f592cae108b96369caded46f15631a60f1cd22d033cdd46905c3ce0ee18"
expected_files = json.loads('{"requirements.txt": "d3057422eeabe167fdc242f31c5038add6e0aa9aa73fb31227eb92b688cbb553", "src/ewb_methods_client/__init__.py": "faddf128f002243598cb56ab227945e50a2c6219453f24a2c633bf32c7ccf4ab", "src/methods_notebooks/__init__.py": "94e70a66bebea6221748a48192079f14297c552697aebc78205fd49128f9a345", "src/methods_notebooks/session.py": "86f788c33cc0db110ae1ae41f5cd4588af7029d8cd3aa935e491cabf6af40b3e", "src/methods_notebooks/setup.py": "17990abbd57a1472a44071834a7364604b452694605e6e4b19624ef8377e6d9f"}')
helper_path = pathlib.Path(os.environ.get("EWB_NOTEBOOK_HELPERS_ZIP", helper_name)).expanduser()
if helper_path.is_file():
    helper_bytes = helper_path.read_bytes()
else:
    helper_url = "https://github.com/sherandoresearch/power-markets-notebooks/releases/download/notebooks-v1.43.0/" + helper_name
    # Only the public, checksum-pinned transport helpers are downloaded here.
    # Inputs and API credentials are selected in the next cell.
    # Python.org macOS installs may lack a system CA bundle. Use the verified
    # certifi trust store when present; certificate and hostname checks stay on.
    try:
        import certifi
    except ImportError:
        tls_context = ssl.create_default_context()
    else:
        tls_context = ssl.create_default_context(cafile=certifi.where())
    with urllib.request.urlopen(helper_url, timeout=90, context=tls_context) as response:
        helper_bytes = response.read(1_000_001)
if len(helper_bytes) > 1_000_000 or hashlib.sha256(helper_bytes).hexdigest() != expected_sha256:
    raise ValueError("Notebook helper release checksum mismatch")
with zipfile.ZipFile(io.BytesIO(helper_bytes)) as helper_zip:
    members = helper_zip.infolist()
    if len(members) != len(expected_files) or {m.filename for m in members} != set(expected_files):
        raise ValueError("Unexpected helper archive contents")
    if any(stat.S_IFMT(m.external_attr >> 16) not in (0, stat.S_IFREG) or
           m.flag_bits & 1 or m.file_size > 100_000 for m in members):
        raise ValueError("Unsupported helper archive member")
    helper_contents = {name: helper_zip.read(name) for name in expected_files}
if any(hashlib.sha256(value).hexdigest() != expected_files[name] for name, value in helper_contents.items()):
    raise ValueError("Notebook helper member checksum mismatch")
requirements = helper_contents["requirements.txt"].decode().splitlines()
missing = []
for requirement in requirements:
    package, version = requirement.split("==")
    try:
        current_version = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        current_version = None
    if current_version != version:
        if package in sys.modules:
            raise RuntimeError("Restart in a fresh kernel before installing " + requirement)
        missing.append(requirement)
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
for requirement in requirements:
    package, version = requirement.split("==")
    if importlib.metadata.version(package) != version:
        raise RuntimeError("Dependency version mismatch: " + package)
if any(name == "methods_notebooks" or name.startswith("methods_notebooks.") or
       name == "ewb_methods_client" or name.startswith("ewb_methods_client.") for name in sys.modules):
    raise RuntimeError("Restart the kernel before rerunning helper setup.")
helper_parent = pathlib.Path.cwd() / "data/processed/notebook-helpers"
helper_parent.mkdir(parents=True, exist_ok=True, mode=0o700)
helper_root = pathlib.Path(tempfile.mkdtemp(prefix="v1.43.0-", dir=helper_parent))
for name, value in helper_contents.items():
    target = helper_root / name
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_bytes(value)
sys.path.insert(0, str(helper_root / "src"))
del helper_bytes, helper_contents
print("Notebook helpers verified. Calculations use Workbench 1.43; inputs and results stay private.")


In [ ]:
from methods_notebooks.setup import configure_notebook
setup_status = configure_notebook('NB-11-ENTSOG', require_key=True)
print(setup_status)


# NB-11-ENTSOG — Observed gas records and collector lineage
A real-data companion to NB-11's lineage and NB-12's vintage distinctions.
**Question:** which result changes because a value was revised, and which changes
because the collector acquired additional hours? We use the reported point
`DIS-00191`, operator `BE-TSO-0001`, keeping entry and exit separate, over three
gas days beginning 10 October 2026. No synthetic observations are introduced.

Work through composite identity, interval alignment, four distinct indicators,
a real collector cutoff, and a specific revised allocation record. The selected
point is a reporting scope, not a participant account or a reconstructed pipeline.

In [ ]:
NOTEBOOK_ID = 'NB-11-ENTSOG'

import copy, json
from datetime import datetime
from types import SimpleNamespace
import numpy as np
from methods_notebooks.session import Session, MethodsError, setup_plot
session = Session(NOTEBOOK_ID, version='1.43.0')
inputs = session.inputs
responses, execution_plan = {}, []
def execute(name, method, request, parents=()):
    request = copy.deepcopy(request)
    if parents:
        request['parent_runs'] = [{k:responses[p]['receipt'][k] for k in
            ('run_id','scientific_method_id','method_version','result_sha256')} for p in parents]
    response = session.execute(method, request)
    responses[name] = response
    execution_plan.append(dict(name=name,method=method,request=request,parents=list(parents),rest_receipt=response['receipt']))
    (session.output/'execution-plan.json').write_text(json.dumps(execution_plan,indent=2,allow_nan=False)+'\n')
    return response['result']
def reject(method, request):
    try:
        session.execute(method, request)
    except MethodsError as exc:
        assert str(exc) == 'Methods request refused (HTTP 422)'
        print('Expected remote rejection:',method,str(exc))
    else:
        raise AssertionError('Invalid retained-input mutation accepted')
plt=setup_plot()

from collections import Counter
indicators=inputs['indicators']
def dt(s):return datetime.fromisoformat(s.replace('Z','+00:00'))
def key(r):return tuple(r[k] for k in inputs['gas_key'])
x=dict(times=[dt(t) for t in inputs['times']],latest={},old={},arrays={},availability={},lags={},audit={},revisions={})
print('Local retained record inspection:',inputs['raw']['latest']['flow'][:1])
remote={name:execute(name,'interval_alignment' if name.endswith(':interval') else 'availability_audit',request) for name,request in inputs['requests'].items()}
for ind in indicators:
    latest={key(r):r for r in inputs['raw']['latest'][ind]};old={key(r):r for r in inputs['raw']['old'][ind]}
    x['latest'][ind]=latest;x['old'][ind]=old
    match=set(latest)&set(old);changed=[k for k in sorted(match) if latest[k]['value']!=old[k]['value']]
    x['revisions'][ind]=[dict(key=k,before=old[k],after=latest[k]) for k in changed]
    x['audit'][ind]=dict(latest_rows=len(latest),cutoff_rows=len(old),common_rows=len(match),value_revisions=len(changed),
        newly_observed_rows=len(set(latest)-set(old)),cutoff_rows_absent_latest=len(set(old)-set(latest)),
        latest_status_counts=dict(Counter(r['flow_status'] or 'not supplied' for r in latest.values())),published_zero_count=sum(r['value']==0 for r in latest.values()))
    x['arrays'][ind]=np.asarray([[remote['latest:'+ind+':'+direction+':interval']['aligned_panel'][i]['value'] for direction in ('entry','exit')] for i in range(72)])
    x['availability'][ind]=np.asarray([[remote['old:'+ind+':'+direction+':interval']['support_ledger'][i]['complete'] for direction in ('entry','exit')] for i in range(72)])
    for direction in ('entry','exit'):assert all(r['complete'] for r in remote['latest:'+ind+':'+direction+':interval']['support_ledger'])
    ledger=remote['old:'+ind+':clock']['availability_ledger'];assert all(r['status']=='available' for r in ledger['records'])
    assert ledger['complete_point_in_time_reconstruction'] is False and ledger['source_claims_verified'] is False
    x['lags'][ind]=np.array([[(dt(r['source_published_utc'])-dt(r['period_end_utc'])).total_seconds()/3600,(dt(r['retrieved_utc'])-dt(r['period_end_utc'])).total_seconds()/3600] for r in latest.values()])
common=set.intersection(*(set(v) for v in x['latest'].values()));prior_common=set.intersection(*(set(v) for v in x['old'].values()))
x['common_count']=len(common);x['prior_common_count']=len(prior_common)
example=sorted(common)[0]
x['trace']=[dict(indicator=ind,**{k:x['latest'][ind][example][k] for k in (*inputs['gas_key'],'unit','value','flow_status','source_published_utc','retrieved_utc','value_hash','payload_sha256','series_id','legacy_series_id')}) for ind in indicators]
print('Indicator    | latest | cutoff | value revisions | added rows | published zeros')
for ind,a in x['audit'].items():print(f"{ind:12} | {a['latest_rows']:6} | {a['cutoff_rows']:6} | {a['value_revisions']:15} | {a['newly_observed_rows']:10} | {a['published_zero_count']:15}")
print('Latest flow status counts:',x['audit']['flow']['latest_status_counts'])
print('Common latest rows:',x['common_count'],'; common cutoff rows:',x['prior_common_count'])

## 1. Identity is more than a point or a hash
Every cross-indicator join uses point, direction, operator, interval start **and
interval end**. Indicator remains a separate coordinate. Four latest feeds each
have 144 rows: 72 hourly intervals × two directions. The retained snapshot is
`entsog-20261015T122505Z-cfb5bf43`, with complete pagination for this point/window.

The initial sampled point DIS-00013 had flow but no nomination, renomination or
allocation rows. We preserved those empty queries, then selected the first
common point in the bounded companion-indicator inventory, DIS-00191, and checked
its flow coverage. Selection is for a teaching example, not a representative
sample of Europe. Other operator reports are never summed into this one.

A value hash is not an entity identifier: equal values can share hashes across
indicators. The legacy series ID is also shared across these feeds; the canonical
series ID includes the indicator. Retain both identity and payload provenance.

In [ ]:
for row in x['trace']:
    print(row['indicator'],row['direction'],row['value'],row['unit'])
    print(' canonical series:',row['series_id'])
    print(' legacy series:',row['legacy_series_id'],'value hash:',row['value_hash'])
assert len({row['series_id'] for row in x['trace']})==4
assert len({row['legacy_series_id'] for row in x['trace']})==1

## 2. Align intervals and preserve the meaning of each feed
The EWB contract defines gas days from 06:00 Europe/Brussels. In this October
sample, that is 04:00 UTC. We verify every start, one-hour end, unit, gas-day
label, and identity before comparing indicators. The window runs from 10 October
04:00 UTC through 13 October 04:00 UTC, excluding the end.

Units are **kWh/h of gas energy**, not electrical MW. Integrate value × 1 hour
to obtain kWh, then divide by 1,000 for MWh of gas energy. No electricity output
or heating-value conversion is inferred. Physical flow, nomination, renomination,
and allocation are distinct reported measures, not interchangeable revisions.
ENTSOG's [publication format](https://transparency.entsog.eu/pdf/TRA158_ENTSOG%20TP%20Data%20Publication%20Format_v3%2017052016.pdf)
distinguishes numeric zero from missing or inapplicable reporting; this export
also has known NA/withdrawal limitations described below.

All 144 nomination and renomination values are **published numeric zeros** in
this selected latest sample. They are not imputed. Their presence does not prove
a meaningful pre-delivery forecast, no demand, or an unutilized pipeline. We show
them as recorded and do not score “forecast error” against physical flow.

In [ ]:
print('Integrated reported measure, MWh of gas energy; entry and exit kept separate')
for ind,a in x['arrays'].items():print(ind,dict(zip(('entry','exit'),np.round(a.sum(0)/1000,3).tolist())))
fig,axes=plt.subplots(2,1,figsize=(11,6),sharex=True)
colors=['#88C0D0','#EBCB8B','#A3BE8C','#B48EAD']
for j,direction in enumerate(('entry','exit')):
    for ind,color in zip(indicators,colors):
        axes[j].plot(x['times'],x['arrays'][ind][:,j]/1000,label=ind,color=color,linestyle='--' if ind=='renomination' else '-')
    axes[j].set(title='Reported '+direction,ylabel='Gas energy rate · MWh/h');axes[j].legend(ncol=4,fontsize=8)
axes[-1].set_xlabel('UTC');fig.autofmt_xdate();fig.tight_layout();plt.show()

## 3. Three clocks, two retained views
Interval time identifies the gas hour. `source_published_utc` carries the TSO's
last-update timestamp. `retrieved_utc` is when the collector first retained that
value. The collector's `as_of` view at **12 October 2026 00:00 UTC** contains 62
flow rows and 96 rows for each other indicator. Latest has 144 in each.

Every cutoff row is checked to have receipt at or before that cutoff. Periods
extending beyond the cutoff can legitimately appear in a reported schedule;
cutoff retrieval alone does not certify that a value is a realized measurement.
We preserve the event and knowledge clocks rather than imposing an invented one.

The scatter below compares source-update and receipt lags relative to interval
end. These timestamps are provenance, not proof of an earlier admissible forecast.
`latest` remains revisable; “latest” must not be relabeled “final.” Half of the
latest physical-flow rows still carry Provisional status in this export.

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(12,4))
for ind,color in zip(indicators,colors):
    lag=x['lags'][ind];axes[0].scatter(lag[:,0],lag[:,1],s=12,alpha=.45,label=ind,color=color)
axes[0].set(xlabel='Source update minus interval end · hours',ylabel='Receipt minus interval end · hours',title='Distinct clocks in latest rows');axes[0].legend(fontsize=8)
mask=np.array([x['availability'][ind][:,j] for ind in indicators for j in range(2)])
from matplotlib.colors import ListedColormap
axes[1].imshow(mask,aspect='auto',interpolation='nearest',cmap=ListedColormap(['#4C566A','#88C0D0']),vmin=0,vmax=1)
axes[1].set_yticks(range(8),[ind+' '+direction for ind in indicators for direction in ('entry','exit')],fontsize=8)
axes[1].set_xticks([0,24,48,71],['Oct 10 04','Oct 11 04','Oct 12 04','Oct 13 03'])
axes[1].set(title='Present at cutoff (blue); unavailable (grey)',xlabel='Interval start · UTC')
fig.tight_layout();plt.show()

## 4. Decompose changed results without changing history
Comparing only common composite keys, 45 allocation values change; the other
three indicators have no value changes on their common keys. Latest also adds
82 flow rows and 48 rows in each other feed. New coverage is not a revision of
an existing value. No row in this retained cutoff set is absent from latest,
but that finding does not remove the service's withdrawal limitation.

For an additive energy summary, the latest-minus-cutoff difference decomposes
into revisions on common keys plus values on newly observed keys, minus any
removed values. This is an accounting identity over two retained query results,
not a return, physical imbalance, or estimator of all historical revisions.

In [ ]:
print('Indicator | revisions on common keys | newly observed contribution | total change (MWh gas)')
for ind in indicators:
    latest=x['latest'][ind];old=x['old'][ind];common=set(latest)&set(old)
    revised=sum(latest[k]['value']-old[k]['value'] for k in common)/1000
    added=sum(latest[k]['value'] for k in set(latest)-set(old))/1000
    removed=sum(old[k]['value'] for k in set(old)-set(latest))/1000
    total=(sum(r['value'] for r in latest.values())-sum(r['value'] for r in old.values()))/1000
    np.testing.assert_allclose(total,revised+added-removed,atol=1e-8)
    print(ind,round(revised,3),round(added,3),round(total,3),sep=' | ')
example=x['revisions']['allocation'][0]
print('One actual revised allocation key:',example['key'])
for name in ('before','after'):
    r=example[name]
    print(name,{k:r[k] for k in ('value','flow_status','source_published_utc','retrieved_utc','value_hash','payload_sha256')})

The decomposition above combines reported entry and exit solely to demonstrate
the additive record ledger. It is **not** a net geographical gas-flow total.
The separate-direction figures retain the economically relevant distinction.
The first allocation revision shown changes a reported value and its status
from Provisional to Confirmed; both versions remain in the private evidence.
Equality of some hashes or values elsewhere does not establish full-history identity.
Value-based capture may not preserve every metadata-only update.

## 5. Negative control and the boundary of historical replay
A real request before the supported collector floor returns the named
`collector_asof_before_valid_from` error. The retained request used 3 October
2026; the advertised floor is 4 October. We do not silently substitute latest.
This establishes the observed service guard, not source-publication as-of support.

The descriptor discloses that NA placeholders are not rows and withdrawals to
NA lack tombstones. Consequently even supported collector-as-of is not a perfect
reconstruction of the source's historical state. Capacity feeds have additional
interval-overlap rules and are not used here. Status, update time, receipt, and
event time must remain separate in any extension.

Exercise: why did a total change when no common flow value changed? Coverage
grew by 82 rows. Why can't zero nominations be scored as an admissible forecast?
Publication meaning and availability are unverified, and the cutoff is collector
time. Why can't you join only on `value_hash` or `legacy_series_id`? Indicators
can share both while reporting different concepts. The audits above supply the
answer scaffold without making up missing observations or earlier timestamps.

In [ ]:
negative=inputs['negative']
assert negative['isError'] and 'collector_asof_before_valid_from' in json.dumps(negative)
print('Observed negative control: collector_asof_before_valid_from; requested 2026-10-03, floor 2026-10-04')

In [ ]:
bad=copy.deepcopy(inputs['requests']['latest:flow:entry:interval']);bad['inputs']['interval_records'].append(copy.deepcopy(bad['inputs']['interval_records'][0]));reject('interval_alignment',bad)
bad=copy.deepcopy(inputs['requests']['latest:flow:entry:interval']);bad['inputs']['alignment_policy']['quantity_kind']='average_power';reject('interval_alignment',bad)
def plain(v):
    if isinstance(v,np.ndarray):return v.tolist()
    if isinstance(v,datetime):return v.isoformat()
    raise TypeError(type(v))
summary_data={k:v for k,v in x.items() if k not in ('latest','old')}
(session.output/'consumer-summary.json').write_text(json.dumps(summary_data,default=plain,allow_nan=False,indent=2)+'\n')
summary=session.finish({'observed_only':True,'all_gas_intervals_preserved':True,'gas_energy_units_preserved':True,'collector_cutoff_verified':True,'published_zeros_not_imputed':True,'no_complete_source_history_claim':True,'new_synthetic_data':False})